# 🚀 FL Shapley 20 Newsgroups — 完整配置与实验运行
# FL Shapley 20 Newsgroups — Full Setup & Experiment Runner

按顺序从上到下运行本 Notebook，即可自动完成：

1. 环境检测与依赖安装
2. 设备自动选择（CPU / CUDA / Apple MPS）
3. 数据加载与分布可视化
4. 运行三组实验（clean / freerider / poisoning）
5. 结果可视化与分析

---

Run each cell top-to-bottom. The notebook will:
1. Check & install dependencies
2. Auto-detect the best device (CPU / CUDA / Apple MPS)
3. Load & visualise the 20 Newsgroups data
4. Run all three FL experiments
5. Display all result plots inline

## 0. 系统信息 / System Information

In [ ]:
import sys, platform
print(f'Python  : {sys.version}')
print(f'OS      : {platform.system()} {platform.release()}')
print(f'Machine : {platform.machine()}')

## 1. 安装依赖 / Install Dependencies

该 cell 会自动检测平台并安装正确版本的 PyTorch。

This cell detects your platform and installs the correct PyTorch build.

| Platform | PyTorch build |
|---|---|
| macOS Apple Silicon (M1/M2/M3/M4) | default wheel (MPS support) |
| Linux/Windows + NVIDIA GPU | CUDA 12.1 wheel |
| Everything else | CPU-only wheel |

In [ ]:
import subprocess, sys, platform

def run(cmd):
    print(f'  $ {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-800:])
    else:
        # show last 3 lines of stdout
        lines = [l for l in result.stdout.strip().split('\n') if l]
        for l in lines[-3:]: print(' ', l)

# ── Detect platform ──────────────────────────────────────────────────────────
system  = platform.system()   # Darwin | Linux | Windows
machine = platform.machine()  # arm64  | x86_64 | AMD64

print('=== Installing core dependencies ===')
run(f'{sys.executable} -m pip install -q numpy pandas scipy scikit-learn matplotlib pyyaml tqdm')

# ── PyTorch install ──────────────────────────────────────────────────────────
print('\n=== Installing PyTorch ===')

# Try to import torch first – skip if already installed with a compatible version
try:
    import torch
    print(f'  torch {torch.__version__} already installed – skipping.')
except ImportError:
    if system == 'Darwin' and machine == 'arm64':
        # Apple Silicon – default wheel includes MPS
        print('  Detected Apple Silicon → installing default PyTorch (MPS enabled)')
        run(f'{sys.executable} -m pip install -q torch')
    elif system == 'Darwin':
        # Intel Mac
        print('  Detected Intel Mac → installing CPU PyTorch')
        run(f'{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu')
    else:
        # Linux / Windows: check for NVIDIA GPU
        nvidia = subprocess.run('nvidia-smi', shell=True, capture_output=True)
        if nvidia.returncode == 0:
            print('  NVIDIA GPU detected → installing CUDA 12.1 PyTorch')
            run(f'{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cu121')
        else:
            print('  No GPU detected → installing CPU PyTorch')
            run(f'{sys.executable} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu')

print('\nAll dependencies ready ✓')

## 2. 设备检测 / Device Detection

自动选择最佳计算设备：CUDA（NVIDIA）> MPS（Apple Silicon）> CPU

Auto-selects the best available device.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
from src.device_utils import print_device_summary, get_device

DEVICE = print_device_summary()
print(f'\nUsing device: {DEVICE}')

## 3. 配置 / Configuration

可在这里修改实验参数。

Edit experiment parameters here before running.

In [ ]:
from src.utils import load_config, set_seed
import yaml, pprint

# Load default config
cfg = load_config('../config.yaml')

# ── Optional overrides ──────────────────────────────────────────────────────
# Uncomment and edit any line to override the default value:

# cfg['num_rounds']              = 20     # communication rounds
# cfg['num_clients']             = 10     # total clients
# cfg['client_fraction']         = 0.5   # fraction selected per round
# cfg['local_epochs']            = 3     # local SGD epochs
# cfg['shapley_num_permutations']= 10    # MC samples for Shapley
# cfg['dirichlet_alpha']         = 0.1   # non-IID degree (smaller = more skewed)
# cfg['free_rider_ratio']        = 0.2   # fraction of free-rider clients
# cfg['poisoning_ratio']         = 0.2   # fraction of poisoning clients
# cfg['device']                  = 'auto'  # 'auto' | 'mps' | 'cuda' | 'cpu'

cfg['output_dir'] = '../outputs'  # save outputs in project root

set_seed(cfg['random_seed'])

print('Active configuration:')
pprint.pprint(cfg)

## 4. 数据加载 / Load & Vectorise Data

In [ ]:
from src.data_utils import load_dataset

X_train, X_val, X_test, y_train, y_val, y_test, vectorizer, class_names = load_dataset(
    test_size=cfg['test_size'],
    val_size=cfg['val_size'],
    max_tfidf_features=cfg['max_tfidf_features'],
    random_seed=cfg['random_seed'],
)

print(f'\nClass names ({len(class_names)}):')
for i, n in enumerate(class_names):
    print(f'  {i:>2}: {n}')

## 5. 数据分区 / Dirichlet Partition & Distribution Visualisation

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from src.partition import dirichlet_partition, save_client_distribution

clients_data = dirichlet_partition(
    X_train, y_train,
    num_clients=cfg['num_clients'],
    alpha=cfg['dirichlet_alpha'],
    random_seed=cfg['random_seed'],
)

# Save distribution CSV
import os
os.makedirs('../outputs', exist_ok=True)
save_client_distribution(clients_data, class_names, '../outputs/client_data_distribution.csv')

# Build heatmap matrix
mat = np.zeros((cfg['num_clients'], len(class_names)))
for c in clients_data:
    for cls, cnt in c['class_counts'].items():
        mat[c['client_id'], cls] = cnt

fig, ax = plt.subplots(figsize=(16, 4))
im = ax.imshow(mat, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(cfg['num_clients']))
ax.set_yticklabels([f'Client {i}' for i in range(cfg['num_clients'])], fontsize=9)
ax.set_title(f'Client Data Distribution  (Dirichlet α={cfg["dirichlet_alpha"]})', fontsize=13)
plt.colorbar(im, ax=ax, label='Sample Count')
plt.tight_layout()
plt.show()
print('Distribution saved to outputs/client_data_distribution.csv')

## 6A. 实验A — Clean 基线 / Experiment A — Clean Baseline

所有客户端诚实训练，建立基准 Shapley 贡献模式。

All clients train honestly. Establishes baseline Shapley patterns.

In [ ]:
import sys; sys.path.insert(0, os.path.abspath('..'))
from src.utils import get_logger, experiment_output_dir
from main import run_experiment

logger = get_logger('nb_clean')
out_clean = experiment_output_dir(cfg['output_dir'], 'clean')
save_client_distribution(clients_data, class_names,
                         os.path.join(out_clean, 'client_data_distribution.csv'))

res_clean = run_experiment(
    cfg=cfg, attack_type='clean',
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_train=y_train, y_val=y_val, y_test=y_test,
    class_names=class_names, clients_data=clients_data,
    output_dir=out_clean, logger=logger, device=DEVICE,
)
print('\nExperiment A complete ✓')

In [ ]:
# Inline accuracy plot
df = res_clean['round_metrics_df']
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(df['round'], df['global_accuracy'], 'o-', label='val acc', color='#2196F3')
ax.plot(df['round'], df['test_accuracy'],   's--', label='test acc', color='#2196F3', alpha=0.6)
ax.set_xlabel('Round'); ax.set_ylabel('Accuracy')
ax.set_title('Clean – Accuracy vs Round'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Show saved Shapley heatmap
from IPython.display import Image
Image(os.path.join(out_clean, 'shapley_heatmap_clean.png'))

## 6B. 实验B — 搭便车攻击 / Experiment B — Free-rider Attack

部分客户端不做真实训练，上传假参数更新。

A fraction of clients skip real training and upload fake parameter updates.

In [ ]:
logger_fr = get_logger('nb_freerider')
out_fr = experiment_output_dir(cfg['output_dir'], 'freerider')
save_client_distribution(clients_data, class_names,
                         os.path.join(out_fr, 'client_data_distribution.csv'))

res_fr = run_experiment(
    cfg=cfg, attack_type='freerider',
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_train=y_train, y_val=y_val, y_test=y_test,
    class_names=class_names, clients_data=clients_data,
    output_dir=out_fr, logger=logger_fr, device=DEVICE,
)
print('\nExperiment B complete ✓')

In [ ]:
df_fr = res_fr['round_metrics_df']
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(df['round'], df['global_accuracy'],    'o-',  label='clean', color='#2196F3')
ax.plot(df_fr['round'], df_fr['global_accuracy'], 'o-', label='freerider', color='#FF9800')
ax.set_xlabel('Round'); ax.set_ylabel('Val Accuracy')
ax.set_title('Clean vs Free-rider'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
Image(os.path.join(out_fr, 'shapley_heatmap_freerider.png'))

## 6C. 实验C — 投毒攻击 / Experiment C — Poisoning Attack

部分客户端在本地训练前翻转标签。

A fraction of clients flip labels locally before training.

In [ ]:
logger_po = get_logger('nb_poisoning')
out_po = experiment_output_dir(cfg['output_dir'], 'poisoning')
save_client_distribution(clients_data, class_names,
                         os.path.join(out_po, 'client_data_distribution.csv'))

res_po = run_experiment(
    cfg=cfg, attack_type='poisoning',
    X_train=X_train, X_val=X_val, X_test=X_test,
    y_train=y_train, y_val=y_val, y_test=y_test,
    class_names=class_names, clients_data=clients_data,
    output_dir=out_po, logger=logger_po, device=DEVICE,
)
print('\nExperiment C complete ✓')

In [ ]:
Image(os.path.join(out_po, 'shapley_heatmap_poisoning.png'))

## 7. 三组实验对比 / Combined Comparison

In [ ]:
from src.plotting import plot_accuracy_vs_round

metrics_dfs = {
    'clean':     res_clean['round_metrics_df'],
    'freerider': res_fr['round_metrics_df'],
    'poisoning': res_po['round_metrics_df'],
}

# Save combined chart
plot_accuracy_vs_round(metrics_dfs, '../outputs/accuracy_vs_round.png')
Image('../outputs/accuracy_vs_round.png')

In [ ]:
# Final accuracy table
import pandas as pd
rows = []
for label, res in [('clean', res_clean), ('freerider', res_fr), ('poisoning', res_po)]:
    last = res['round_metrics_df'].iloc[-1]
    rows.append({'Experiment': label,
                 'Val Accuracy': f"{last['global_accuracy']:.4f}",
                 'Val Macro-F1': f"{last['global_macro_f1']:.4f}",
                 'Test Accuracy': f"{last['test_accuracy']:.4f}"})
pd.DataFrame(rows)

## 8. Shapley 类贡献对比 / Class-Level Shapley Comparison

In [ ]:
shapley_dfs = {
    'clean':     res_clean['shapley_df'],
    'freerider': res_fr['shapley_df'],
    'poisoning': res_po['shapley_df'],
}

COLOURS = {'clean': '#2196F3', 'freerider': '#FF9800', 'poisoning': '#F44336'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (attack, df) in zip(axes, shapley_dfs.items()):
    if df is None or df.empty:
        ax.set_visible(False); continue
    mean_sv = df.groupby('class_name')['shapley_value'].mean().sort_values()
    colours = [('#F44336' if v < 0 else COLOURS[attack]) for v in mean_sv.values]
    ax.barh(mean_sv.index, mean_sv.values, color=colours, edgecolor='white')
    ax.axvline(0, color='k', lw=0.8)
    ax.set_title(f'Mean Shapley – {attack}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean Shapley Value', fontsize=9)
    ax.tick_params(axis='y', labelsize=7)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

## 9. 可疑客户端检测 / Suspicious Client Detection

In [ ]:
THRESHOLD = 0.005
print(f'Clients with |mean Shapley| < {THRESHOLD}  →  possibly free-riders or poisoners\n')

for attack, df in shapley_dfs.items():
    if df is None or df.empty: continue
    stats = df.groupby(['client_id','client_role'])['shapley_value'].agg(
        mean='mean',
        mean_abs=lambda x: abs(x).mean()
    ).reset_index()
    sus = stats[stats['mean_abs'] < THRESHOLD]
    print(f'=== {attack.upper()} ===')
    if sus.empty:
        print('  No suspicious clients found.')
    else:
        print(sus.round(6).to_string(index=False))
    print()

## 10. Top-5 最高 / 最低贡献类 / Top-5 Best & Worst Contributing Classes

In [ ]:
sv_clean = shapley_dfs['clean']
if sv_clean is not None and not sv_clean.empty:
    mean_sv = sv_clean.groupby('class_name')['shapley_value'].mean().sort_values(ascending=False)
    
    print('Top 5 most contributive classes (clean):')
    for cls, v in mean_sv.head(5).items():
        print(f'  {cls:<45s}  {v:+.5f}')
    
    print('\nBottom 5 least contributive classes (clean):')
    for cls, v in mean_sv.tail(5).items():
        print(f'  {cls:<45s}  {v:+.5f}')

## 11. 保存路径总结 / Output File Summary

In [ ]:
import glob
output_files = sorted(glob.glob('../outputs/**/*', recursive=True))
files = [f for f in output_files if os.path.isfile(f)]
print(f'Generated {len(files)} output files:\n')
for f in files:
    size = os.path.getsize(f)
    unit = 'KB' if size < 1_000_000 else 'MB'
    val  = size // 1024 if size < 1_000_000 else size // (1024*1024)
    rel  = os.path.relpath(f, '..')
    print(f'  {rel:<60s} {val:>5} {unit}')

---
## ✅ 完成 / Done!

所有输出已保存到 `outputs/` 目录。

All outputs saved to the `outputs/` directory.

下一步 / Next steps:
- 用 Excel 打开 `outputs/*/class_shapley_by_round.csv` 查看详细的 round × client × class Shapley 表格
- 运行 `python analysis.py` 获取文字分析摘要
- 修改 `config.yaml` 中的超参数后重新运行实验